In [1]:
import os
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded")
print(f"Model: all-MiniLM-L6-v2")

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded
Model: all-MiniLM-L6-v2


In [2]:
sentence = "Wire transfer fraud detected at 3am"

embedding = model.encode(sentence)

print(f"Text: '{sentence}'")
print(f"\nEmbedding type: {type(embedding)}")
print(f"Embedding length: {len(embedding)}")
print(f"\nFirst 10 numbers: {embedding[:10]}")
print(f"Last 10 numbers:  {embedding[-10:]}")

Text: 'Wire transfer fraud detected at 3am'

Embedding type: <class 'numpy.ndarray'>
Embedding length: 384

First 10 numbers: [-0.05498111  0.02587106 -0.01175341 -0.02671981  0.03137748 -0.04071675
  0.09033709 -0.01973968  0.03872918  0.00391432]
Last 10 numbers:  [ 0.04444652 -0.07299091 -0.01120033 -0.02098807 -0.0553886  -0.01595229
  0.01233902 -0.04264937 -0.078068   -0.02186571]


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# Three sentences
sentence1 = "Wire transfer fraud detected at 3am"
sentence2 = "Telegraphic transfer suspicious activity at night"
sentence3 = "Recipe for chocolate cake with butter"

# Create embeddings for all three
emb1 = model.encode(sentence1)
emb2 = model.encode(sentence2)
emb3 = model.encode(sentence3)

# Compare cosine_similarity
sim_1_2 = cosine_similarity([emb1], [emb2])[0][0]
sim_1_3 = cosine_similarity([emb1], [emb3])[0][0]
sim_2_3 = cosine_similarity([emb2], [emb3])[0][0]

print("=== Cosine Similarity Scores ===\n")
print(f"Sentence 1: '{sentence1}'")
print(f"Sentence 2: '{sentence2}'")
print(f"Sentence 3: '{sentence3}'")

print(f"\nSimilarity 1 vs 2: {sim_1_2:.4f}")
print(f"Similarity 1 vs 3: {sim_1_3:.4f}")
print(f"Similarity 2 vs 3: {sim_2_3:.4f}")

=== Cosine Similarity Scores ===

Sentence 1: 'Wire transfer fraud detected at 3am'
Sentence 2: 'Telegraphic transfer suspicious activity at night'
Sentence 3: 'Recipe for chocolate cake with butter'

Similarity 1 vs 2: 0.6288
Similarity 1 vs 3: -0.0652
Similarity 2 vs 3: -0.0706


In [4]:
print("\n=== Interpretation ===")
print(f"1 vs 2: {sim_1_2:.4f} → ", end="")
if sim_1_2 > 0.7:
    print("VERY SIMILAR meaning ✅")
elif sim_1_2 > 0.4:
    print("MODERATELY SIMILAR meaning")
else:
    print("DIFFERENT meaning")

print(f"1 vs 3: {sim_1_3:.4f} → ", end="")
if sim_1_3 > 0.7:
    print("VERY SIMILAR meaning")
elif sim_1_3 > 0.4:
    print("MODERATELY SIMILAR meaning")
else:
    print("DIFFERENT meaning ✅")


=== Interpretation ===
1 vs 2: 0.6288 → MODERATELY SIMILAR meaning
1 vs 3: -0.0652 → DIFFERENT meaning ✅


In [5]:
# ============================================================
# Document Search Using Embeddings
# Simulating how RAG finds relevant documents
# ============================================================

# These are your "knowledge base" documents
# Like pages from a bank policy manual
documents = [
    "Wire transfer fraud limit is $10,000 per day. "
    "Transactions above this require additional verification.",
    
    "Credit card fraud is detected using machine learning "
    "models that analyze spending patterns and flag anomalies.",
    
    "Telegraphic transfers above $5,000 to high-risk countries "
    "require compliance approval before processing.",
    
    "Customer KYC documents must be verified annually. "
    "Missing documents trigger account restrictions.",
    
    "ESG investment restrictions are reviewed quarterly. "
    "Portfolio managers must comply within 30 days of notification.",
    
    "Unauthorized access to customer accounts must be reported "
    "to the fraud team within 24 hours of detection.",
    
    "Chocolate cake requires flour, butter, eggs and sugar. "
    "Bake at 180 degrees for 35 minutes."
]

# Step 1: Create embeddings for ALL documents
print("Creating embeddings for all documents...")
doc_embeddings = model.encode(documents)
print(f"Created {len(doc_embeddings)} embeddings")
print(f"Each embedding: {len(doc_embeddings[0])} numbers\n")

# Step 2: Search function
def find_relevant_docs(query, top_k=2):
    """Find most relevant documents for a query"""
    
    # Embed the query
    query_embedding = model.encode(query)
    
    # Compare query to all documents
    similarities = cosine_similarity(
        [query_embedding], 
        doc_embeddings
    )[0]
    
    # Get top_k most similar documents
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    print(f"Query: '{query}'")
    print(f"\nTop {top_k} relevant documents found:\n")
    
    for rank, idx in enumerate(top_indices, 1):
        print(f"Rank {rank} (similarity: {similarities[idx]:.4f}):")
        print(f"  {documents[idx][:100]}...")
        print()
    
    return [documents[i] for i in top_indices]

# Test searches
print("=" * 50)
relevant = find_relevant_docs(
    "What is the limit for wire transfer fraud?"
)

print("=" * 50)
relevant2 = find_relevant_docs(
    "How quickly must unauthorized account access be reported?"
)

Creating embeddings for all documents...
Created 7 embeddings
Each embedding: 384 numbers

Query: 'What is the limit for wire transfer fraud?'

Top 2 relevant documents found:

Rank 1 (similarity: 0.8901):
  Wire transfer fraud limit is $10,000 per day. Transactions above this require additional verificatio...

Rank 2 (similarity: 0.5602):
  Telegraphic transfers above $5,000 to high-risk countries require compliance approval before process...

Query: 'How quickly must unauthorized account access be reported?'

Top 2 relevant documents found:

Rank 1 (similarity: 0.7621):
  Unauthorized access to customer accounts must be reported to the fraud team within 24 hours of detec...

Rank 2 (similarity: 0.2872):
  Customer KYC documents must be verified annually. Missing documents trigger account restrictions....



#### Key learnings:
- Embeddings convert text to 384 numbers capturing MEANING
- Similar meanings → similar numbers → high cosine score
- Different meanings → different numbers → low/negative score
- Embedding search finds relevant documents even when 
  exact words don't match (telegraphic = wire transfer)

## Vector Databases with ChromaDB

So far: manually created embeddings and searched them
Problem: doesn't scale to thousands of documents

Solution: ChromaDB — a proper vector database
- Stores embeddings permanently
- Searches efficiently at scale
- Handles metadata and filtering
- Free and runs locally

In [6]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path="./banking_knowledge_base"
)

collection = chroma_client.get_or_create_collection(
    name="banking_policies",
    metadata={"description": "Bank policy documents"}
)

print("ChromaDB ready")
print(f"Collection: banking_policies")
print(f"Documents stored: {collection.count()}")

ChromaDB ready
Collection: banking_policies
Documents stored: 0


In [8]:
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))] 
)

print(f"Documents added: {len(documents)}")
print(f"Total in collection: {collection.count()}")

C:\Users\Erappa Monnanda\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [06:04<00:00, 228kiB/s]   


Documents added: 7
Total in collection: 7
